# Use Case 03 — Customer Churn Prediction Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/usecases/notebooks/03-churn-prediction.ipynb)

Build an end-to-end churn prediction system for a SaaS company. This notebook generates
synthetic customer data, engineers predictive features, trains an XGBoost classifier with
hyperparameter tuning, evaluates model fairness, and produces ranked intervention lists.

**GCP Services Covered:** Vertex AI Pipelines, Vertex AI Vizier (HPT), BigQuery, Model Monitoring, XGBoost Serving Containers

---

## 1. Setup & Installations

In [ ]:
# Install required packages
!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, average_precision_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.facecolor'] = '#0b0d14'
plt.rcParams['figure.facecolor'] = '#0b0d14'
plt.rcParams['text.color'] = '#e6e8f4'
plt.rcParams['axes.labelcolor'] = '#8e94b5'
plt.rcParams['xtick.color'] = '#8e94b5'
plt.rcParams['ytick.color'] = '#8e94b5'
plt.rcParams['axes.edgecolor'] = '#232840'
plt.rcParams['grid.color'] = '#1a1f34'

SEED = 42
np.random.seed(SEED)

print('All packages loaded successfully.')

## 2. Generate Synthetic SaaS Customer Data

We simulate 5,000 SaaS customers with realistic usage, billing, support, and feature
adoption data. The churn label is generated based on a combination of these signals,
mirroring real-world churn patterns.

In [ ]:
N_CUSTOMERS = 5000

# --- Customer demographics ---
customer_ids = [f'CUST_{i:05d}' for i in range(N_CUSTOMERS)]
plan_types = np.random.choice(
    ['free', 'starter', 'professional', 'enterprise'],
    size=N_CUSTOMERS,
    p=[0.15, 0.35, 0.35, 0.15]
)
tenure_months = np.random.exponential(scale=18, size=N_CUSTOMERS).astype(int)
tenure_months = np.clip(tenure_months, 1, 72)

company_sizes = np.random.choice(
    ['1-10', '11-50', '51-200', '201-1000', '1000+'],
    size=N_CUSTOMERS,
    p=[0.20, 0.30, 0.25, 0.15, 0.10]
)

# MRR based on plan type
mrr_map = {'free': 0, 'starter': 49, 'professional': 149, 'enterprise': 499}
base_mrr = np.array([mrr_map[p] for p in plan_types], dtype=float)
mrr = base_mrr * np.random.uniform(0.8, 1.5, N_CUSTOMERS)  # some variance
mrr = np.round(mrr, 2)

# --- Usage features (30-day window) ---
# Base engagement varies by plan type
plan_engagement = {'free': 0.3, 'starter': 0.5, 'professional': 0.7, 'enterprise': 0.85}
base_engagement = np.array([plan_engagement[p] for p in plan_types])

active_days_30d = np.random.binomial(30, base_engagement * np.random.uniform(0.5, 1.2, N_CUSTOMERS))
active_days_30d = np.clip(active_days_30d, 0, 30)

avg_session_seconds = np.random.exponential(scale=600, size=N_CUSTOMERS) * base_engagement
avg_session_seconds = np.clip(avg_session_seconds, 0, 3600).astype(int)

total_events_30d = np.random.poisson(lam=50 * base_engagement, size=N_CUSTOMERS)
features_used = np.random.binomial(12, base_engagement * 0.8, N_CUSTOMERS)
features_used = np.clip(features_used, 0, 12)

days_since_last_active = np.random.exponential(scale=5 / base_engagement, size=N_CUSTOMERS).astype(int)
days_since_last_active = np.clip(days_since_last_active, 0, 30)

# --- Usage trends ---
# Negative trend = declining engagement
wow_event_change = np.random.normal(loc=0.0, scale=0.3, size=N_CUSTOMERS)
wow_duration_change = np.random.normal(loc=0.0, scale=0.25, size=N_CUSTOMERS)
usage_trend_corr = np.random.uniform(-1, 1, size=N_CUSTOMERS)

# --- Support features (30-day window) ---
support_tickets_30d = np.random.poisson(lam=1.5, size=N_CUSTOMERS)
critical_tickets_30d = np.random.binomial(support_tickets_30d, 0.15)
open_tickets = np.random.binomial(support_tickets_30d, 0.3)
avg_resolution_hours = np.random.exponential(scale=24, size=N_CUSTOMERS)
avg_sentiment = np.random.normal(loc=0.1, scale=0.4, size=N_CUSTOMERS)
avg_sentiment = np.clip(avg_sentiment, -1, 1)
bug_tickets = np.random.binomial(support_tickets_30d, 0.4)
billing_tickets = np.random.binomial(support_tickets_30d, 0.1)

# --- Billing features (90-day window) ---
failed_payments_90d = np.random.poisson(lam=0.3, size=N_CUSTOMERS)
downgrades_90d = np.random.binomial(1, 0.08, N_CUSTOMERS)
discounts_received = np.random.binomial(2, 0.15, N_CUSTOMERS)

# Monthly-only customers are more churn-prone
is_monthly = (plan_types != 'enterprise').astype(int)
contract_months_remaining = np.where(
    plan_types == 'enterprise',
    np.random.randint(0, 12, N_CUSTOMERS),
    0
)

print(f'Generated {N_CUSTOMERS} synthetic customers')

In [ ]:
# --- Generate churn labels based on realistic signal combinations ---

# Compute a latent churn propensity score
churn_score = (
    -0.05 * active_days_30d          # More active = less churn
    + 0.08 * days_since_last_active   # Inactive = more churn
    - 0.03 * features_used            # More features = stickier
    - 0.4 * wow_event_change          # Declining usage = churn
    - 0.3 * usage_trend_corr          # Negative trend = churn
    + 0.15 * support_tickets_30d      # More tickets = frustration
    + 0.5 * critical_tickets_30d      # Critical issues = high churn
    + 0.3 * open_tickets              # Unresolved = bad
    - 0.5 * avg_sentiment             # Negative sentiment = churn
    + 0.8 * billing_tickets           # Billing issues = very strong signal
    + 0.6 * failed_payments_90d       # Failed payments = churn
    + 1.2 * downgrades_90d            # Downgrade = strong signal
    - 0.015 * tenure_months           # Longer tenure = less churn
    + 0.3 * is_monthly                # Monthly plans churn more
    - 0.001 * mrr                     # Higher MRR = more invested
)

# Add noise
churn_score += np.random.normal(0, 0.5, N_CUSTOMERS)

# Convert to probability via sigmoid
churn_prob = 1 / (1 + np.exp(-churn_score))

# Sample churn labels (target ~6% churn rate)
# Adjust threshold to hit target rate
threshold_adjustment = np.percentile(churn_prob, 94)  # top 6%
churned = (churn_prob > threshold_adjustment).astype(int)

actual_churn_rate = churned.mean()
print(f'Churn rate: {actual_churn_rate:.1%} ({churned.sum()} churned out of {N_CUSTOMERS})')

In [ ]:
# Assemble into a DataFrame
df = pd.DataFrame({
    'customer_id': customer_ids,
    'plan_type': plan_types,
    'company_size': company_sizes,
    'tenure_months': tenure_months,
    'mrr': mrr,
    'active_days_30d': active_days_30d,
    'avg_session_seconds': avg_session_seconds,
    'total_events_30d': total_events_30d,
    'features_used': features_used,
    'days_since_last_active': days_since_last_active,
    'wow_event_change': np.round(wow_event_change, 4),
    'wow_duration_change': np.round(wow_duration_change, 4),
    'usage_trend_corr': np.round(usage_trend_corr, 4),
    'support_tickets_30d': support_tickets_30d,
    'critical_tickets_30d': critical_tickets_30d,
    'open_tickets': open_tickets,
    'avg_resolution_hours': np.round(avg_resolution_hours, 1),
    'avg_sentiment': np.round(avg_sentiment, 3),
    'bug_tickets': bug_tickets,
    'billing_tickets': billing_tickets,
    'failed_payments_90d': failed_payments_90d,
    'downgrades_90d': downgrades_90d,
    'discounts_received': discounts_received,
    'is_monthly_plan': is_monthly,
    'contract_months_remaining': contract_months_remaining,
    'churned': churned
})

print(f'DataFrame shape: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Churn distribution
churn_counts = df['churned'].value_counts()
colors = ['#22c55e', '#ef4444']
axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=colors, edgecolor='#232840')
axes[0].set_title('Churn Distribution', color='#e6e8f4', fontsize=14, fontweight='bold')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 20, f'{v} ({v/len(df):.1%})', ha='center', color='#e6e8f4', fontsize=11)

# 2. Churn by plan type
churn_by_plan = df.groupby('plan_type')['churned'].mean().sort_values(ascending=False)
plan_colors = ['#f59e0b' if v > 0.06 else '#6366f1' for v in churn_by_plan.values]
axes[1].barh(churn_by_plan.index, churn_by_plan.values, color=plan_colors, edgecolor='#232840')
axes[1].set_title('Churn Rate by Plan Type', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn Rate')
for i, v in enumerate(churn_by_plan.values):
    axes[1].text(v + 0.002, i, f'{v:.1%}', va='center', color='#e6e8f4', fontsize=11)

# 3. Churn by company size
churn_by_size = df.groupby('company_size')['churned'].mean().sort_values(ascending=False)
size_colors = ['#f59e0b' if v > 0.06 else '#22d3ee' for v in churn_by_size.values]
axes[2].barh(churn_by_size.index, churn_by_size.values, color=size_colors, edgecolor='#232840')
axes[2].set_title('Churn Rate by Company Size', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Churn Rate')
for i, v in enumerate(churn_by_size.values):
    axes[2].text(v + 0.002, i, f'{v:.1%}', va='center', color='#e6e8f4', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Key feature distributions: churned vs retained
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features_to_plot = [
    ('active_days_30d', 'Active Days (30d)'),
    ('days_since_last_active', 'Days Since Last Active'),
    ('features_used', 'Features Used'),
    ('support_tickets_30d', 'Support Tickets (30d)'),
    ('avg_sentiment', 'Avg Support Sentiment'),
    ('tenure_months', 'Tenure (months)')
]

for ax, (col, title) in zip(axes.flat, features_to_plot):
    retained = df[df['churned'] == 0][col]
    churned_vals = df[df['churned'] == 1][col]
    ax.hist(retained, bins=25, alpha=0.6, color='#22c55e', label='Retained', density=True)
    ax.hist(churned_vals, bins=25, alpha=0.6, color='#ef4444', label='Churned', density=True)
    ax.set_title(title, color='#e6e8f4', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions: Churned vs Retained',
             color='#e6e8f4', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features with churn
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_churn = df[numeric_cols].corr()['churned'].drop('churned').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#ef4444' if v > 0 else '#22c55e' for v in corr_with_churn.values]
ax.barh(corr_with_churn.index, corr_with_churn.values, color=colors, edgecolor='#232840')
ax.set_title('Feature Correlation with Churn', color='#e6e8f4', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='#545a78', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\nTop 5 positive correlations (higher = more churn):')
print(corr_with_churn.tail(5).to_string())
print('\nTop 5 negative correlations (higher = less churn):')
print(corr_with_churn.head(5).to_string())

## 4. Feature Engineering

Create composite features that capture higher-order patterns.
In production, these would be computed in BigQuery SQL.

In [ ]:
df_model = df.copy()

# --- Engagement Score (0-100) ---
# Weighted composite of usage signals
df_model['engagement_score'] = (
    (df_model['active_days_30d'] / 30.0) * 30 +              # Login frequency (30%)
    np.minimum(df_model['avg_session_seconds'] / 1800.0, 1.0) * 25 +  # Session depth (25%)
    (df_model['features_used'] / 12.0) * 25 +                # Feature breadth (25%)
    np.maximum(0, 1 - df_model['days_since_last_active'] / 14.0) * 20  # Recency (20%)
)
df_model['engagement_score'] = np.clip(df_model['engagement_score'], 0, 100).round(2)

# --- Support Health Score ---
# Lower = worse customer health
df_model['support_health_score'] = (
    50
    - df_model['support_tickets_30d'] * 5
    - df_model['critical_tickets_30d'] * 15
    - df_model['open_tickets'] * 10
    + df_model['avg_sentiment'] * 20
    - df_model['billing_tickets'] * 25
)
df_model['support_health_score'] = np.clip(df_model['support_health_score'], 0, 100).round(2)

# --- Billing Risk Score ---
df_model['billing_risk_score'] = (
    df_model['failed_payments_90d'] * 20 +
    df_model['downgrades_90d'] * 30 +
    df_model['billing_tickets'] * 25 +
    (1 - df_model['is_monthly_plan']) * (-10)  # Annual plans = lower risk
)
df_model['billing_risk_score'] = np.clip(df_model['billing_risk_score'], 0, 100).round(2)

# --- Usage Momentum ---
# Combines trend signals into one indicator
df_model['usage_momentum'] = (
    df_model['wow_event_change'] * 0.4 +
    df_model['wow_duration_change'] * 0.3 +
    df_model['usage_trend_corr'] * 0.3
).round(4)

# --- Interaction features ---
df_model['engagement_x_tenure'] = (df_model['engagement_score'] * np.log1p(df_model['tenure_months'])).round(2)
df_model['tickets_per_active_day'] = np.where(
    df_model['active_days_30d'] > 0,
    df_model['support_tickets_30d'] / df_model['active_days_30d'],
    df_model['support_tickets_30d']
).round(4)

print(f'Engineered features. New shape: {df_model.shape}')
print(f'\nNew features added:')
new_features = ['engagement_score', 'support_health_score', 'billing_risk_score',
                'usage_momentum', 'engagement_x_tenure', 'tickets_per_active_day']
print(df_model[new_features].describe().round(2).to_string())

## 5. Train/Test Split with Temporal Considerations

In production, you would split temporally (train on older data, test on recent).
Here we use stratified splitting to preserve class distribution, and note the
temporal approach in comments.

In [ ]:
# Encode categorical features
le_plan = LabelEncoder()
le_size = LabelEncoder()
df_model['plan_type_encoded'] = le_plan.fit_transform(df_model['plan_type'])
df_model['company_size_encoded'] = le_size.fit_transform(df_model['company_size'])

# Define feature columns (exclude identifiers, raw categoricals, target)
exclude_cols = ['customer_id', 'plan_type', 'company_size', 'churned']
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

X = df_model[feature_cols]
y = df_model['churned']

# Stratified split (in production: temporal split)
# In GCP Vertex AI, you'd use:
#   df.sort_values('snapshot_date')
#   train = df[df.snapshot_date < cutoff]
#   test  = df[df.snapshot_date >= cutoff]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples ({y_train.mean():.1%} churn)')
print(f'Test set:     {X_test.shape[0]} samples ({y_test.mean():.1%} churn)')
print(f'Features:     {X_train.shape[1]}')
print(f'\nFeature list:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

## 6. XGBoost with Hyperparameter Tuning

Using GridSearchCV as a local equivalent of Vertex AI Vizier.
In production, Vizier uses Bayesian optimization (Gaussian Process bandits)
for more efficient search over larger parameter spaces.

In [ ]:
# Handle class imbalance
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos
print(f'Class imbalance ratio: {scale_pos_weight:.1f}:1 (negative:positive)')
print(f'Using scale_pos_weight = {scale_pos_weight:.2f}')

# --- Hyperparameter grid (local Vizier equivalent) ---
# In production with Vertex AI Vizier, you'd search a larger space
# using Bayesian optimization with 50-100 trials.
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [200, 300, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'min_child_weight': [3, 5],
}

base_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',  # AUC-PR is better for imbalanced data
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    use_label_encoder=False
)

# Stratified K-Fold for reliable evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='average_precision',  # AUC-PR
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Starting hyperparameter search...')
grid_search.fit(X_train, y_train)

print(f'\nBest parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV AUC-PR: {grid_search.best_score_:.4f}')

In [ ]:
# Use the best model from grid search
model = grid_search.best_estimator_

# Predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print('Model trained and predictions generated.')

## 7. Model Evaluation

Comprehensive evaluation with accuracy, precision, recall, F1, AUC-ROC,
and AUC-PR. For churn prediction, **precision at top-K** and **calibration**
are particularly important.

In [ ]:
# --- Core metrics ---
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba),
    'AUC-PR': average_precision_score(y_test, y_pred_proba)
}

print('=' * 50)
print('MODEL EVALUATION RESULTS')
print('=' * 50)
for name, value in metrics.items():
    print(f'{name:15s}: {value:.4f}')

print(f'\n{"=" * 50}')
print('CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

In [ ]:
# --- Evaluation plots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0, 0].plot(fpr, tpr, color='#f59e0b', linewidth=2,
                label=f'AUC-ROC = {metrics["AUC-ROC"]:.3f}')
axes[0, 0].plot([0, 1], [0, 1], color='#545a78', linestyle='--', linewidth=1)
axes[0, 0].set_title('ROC Curve', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].legend(loc='lower right', fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# 2. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[0, 1].plot(rec, prec, color='#22d3ee', linewidth=2,
                label=f'AUC-PR = {metrics["AUC-PR"]:.3f}')
axes[0, 1].axhline(y=y_test.mean(), color='#545a78', linestyle='--', linewidth=1,
                    label=f'Baseline = {y_test.mean():.3f}')
axes[0, 1].set_title('Precision-Recall Curve', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].legend(loc='upper right', fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            ax=axes[1, 0], cbar_kws={'label': 'Count'},
            annot_kws={'fontsize': 14})
axes[1, 0].set_title('Confusion Matrix', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_xlabel('Predicted')

# 4. Calibration Curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
axes[1, 1].plot(prob_pred, prob_true, color='#a855f7', linewidth=2,
                marker='o', markersize=6, label='Model')
axes[1, 1].plot([0, 1], [0, 1], color='#545a78', linestyle='--', linewidth=1,
                label='Perfectly Calibrated')
axes[1, 1].set_title('Calibration Curve', color='#e6e8f4', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Mean Predicted Probability')
axes[1, 1].set_ylabel('Fraction of Positives')
axes[1, 1].legend(loc='upper left', fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Churn Prediction Model Evaluation',
             color='#e6e8f4', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Feature Importance & SHAP-like Analysis

Understanding which features drive churn predictions is critical for:
1. Business stakeholders to trust and act on predictions
2. GDPR right-to-explanation compliance
3. Identifying data quality issues or leakage

In [ ]:
# --- XGBoost built-in feature importance ---
importance_types = ['weight', 'gain', 'cover']

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for ax, imp_type in zip(axes, importance_types):
    importances = model.get_booster().get_score(importance_type=imp_type)
    # Map feature indices to names
    imp_df = pd.Series(importances).sort_values(ascending=True).tail(15)
    # Try to map f0, f1, ... to actual feature names
    name_map = {f'f{i}': col for i, col in enumerate(feature_cols)}
    imp_df.index = [name_map.get(idx, idx) for idx in imp_df.index]

    colors = ['#f59e0b' if v > imp_df.median() else '#6366f1' for v in imp_df.values]
    ax.barh(imp_df.index, imp_df.values, color=colors, edgecolor='#232840')
    ax.set_title(f'Feature Importance ({imp_type})',
                 color='#e6e8f4', fontsize=13, fontweight='bold')

plt.suptitle('XGBoost Feature Importance (3 methods)',
             color='#e6e8f4', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Permutation-based importance (model-agnostic, SHAP-like) ---
from sklearn.inspection import permutation_importance

print('Computing permutation importance (this may take a moment)...')
perm_result = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=SEED,
    scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

print('\nTop 15 features by permutation importance (AUC-ROC drop):')
print(perm_df.head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top15 = perm_df.head(15).sort_values('importance_mean', ascending=True)
colors = ['#ef4444' if v > 0.01 else '#f59e0b' if v > 0.005 else '#6366f1'
          for v in top15['importance_mean'].values]
ax.barh(top15['feature'], top15['importance_mean'],
        xerr=top15['importance_std'], color=colors, edgecolor='#232840',
        capsize=3, ecolor='#545a78')
ax.set_title('Permutation Feature Importance (AUC-ROC Drop)',
             color='#e6e8f4', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean AUC-ROC Decrease')
plt.tight_layout()
plt.show()

## 9. Churn Risk Scoring & Intervention Ranking

Translate model predictions into actionable intervention priorities.
This mirrors the BigQuery intervention scoring query from the HTML page.

In [ ]:
# --- Build risk scoring table ---
risk_df = df.loc[X_test.index].copy()
risk_df['churn_probability'] = y_pred_proba
risk_df['churn_predicted'] = y_pred
risk_df['actual_churned'] = y_test.values

# Estimated save probability based on tenure and risk level
def estimate_save_prob(row):
    if row['tenure_months'] > 12 and row['churn_probability'] < 0.8:
        return 0.45
    elif row['tenure_months'] > 6 and row['churn_probability'] < 0.6:
        return 0.55
    elif row['churn_probability'] >= 0.9:
        return 0.10
    else:
        return 0.30

risk_df['save_probability'] = risk_df.apply(estimate_save_prob, axis=1)

# Expected annual value saved
risk_df['expected_annual_value_saved'] = (
    risk_df['churn_probability'] *
    risk_df['save_probability'] *
    risk_df['mrr'] * 12
).round(2)

# Intervention tier
def assign_tier(row):
    if row['churn_probability'] >= 0.7 and row['mrr'] >= 500:
        return 'TIER_1_EXECUTIVE'
    elif row['churn_probability'] >= 0.5 and row['mrr'] >= 200:
        return 'TIER_2_CSM'
    elif row['churn_probability'] >= 0.4:
        return 'TIER_3_AUTOMATED'
    else:
        return 'MONITOR'

risk_df['intervention_tier'] = risk_df.apply(assign_tier, axis=1)

# Sort by expected value
risk_df = risk_df.sort_values('expected_annual_value_saved', ascending=False)

print('Intervention Tier Distribution:')
print(risk_df['intervention_tier'].value_counts().to_string())
print(f'\nTotal expected annual value saveable: ${risk_df["expected_annual_value_saved"].sum():,.0f}')

In [ ]:
# --- Top 20 highest-priority customers ---
display_cols = [
    'customer_id', 'plan_type', 'mrr', 'tenure_months',
    'churn_probability', 'save_probability',
    'expected_annual_value_saved', 'intervention_tier'
]

print('TOP 20 CUSTOMERS FOR PROACTIVE OUTREACH')
print('=' * 100)
print(risk_df[display_cols].head(20).to_string(index=False))

In [ ]:
# --- Visualize risk distribution ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Churn probability distribution
axes[0].hist(risk_df['churn_probability'], bins=50, color='#f59e0b',
             edgecolor='#232840', alpha=0.8)
axes[0].axvline(x=0.5, color='#ef4444', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[0].set_title('Churn Probability Distribution',
                  color='#e6e8f4', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Churn Probability')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=10)

# 2. Expected value by tier
tier_value = risk_df.groupby('intervention_tier')['expected_annual_value_saved'].sum()
tier_colors = {'TIER_1_EXECUTIVE': '#ef4444', 'TIER_2_CSM': '#f59e0b',
               'TIER_3_AUTOMATED': '#22d3ee', 'MONITOR': '#545a78'}
colors = [tier_colors.get(t, '#6366f1') for t in tier_value.index]
axes[1].bar(tier_value.index, tier_value.values, color=colors, edgecolor='#232840')
axes[1].set_title('Expected Annual Value by Tier',
                  color='#e6e8f4', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Expected $ Saved')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(tier_value.values):
    axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', color='#e6e8f4', fontsize=10)

# 3. Risk vs Value scatter
axes[2].scatter(
    risk_df['churn_probability'],
    risk_df['mrr'],
    c=risk_df['churn_probability'],
    cmap='YlOrRd', alpha=0.6, s=20, edgecolors='#232840', linewidth=0.3
)
axes[2].set_title('Churn Risk vs Customer Value',
                  color='#e6e8f4', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Churn Probability')
axes[2].set_ylabel('Monthly Recurring Revenue ($)')
# Highlight danger zone
axes[2].axvline(x=0.5, color='#ef4444', linestyle='--', alpha=0.5)
axes[2].axhline(y=200, color='#f59e0b', linestyle='--', alpha=0.5)
axes[2].text(0.75, 220, 'High Risk\nHigh Value', color='#ef4444',
             fontsize=11, fontweight='bold', ha='center')

plt.suptitle('Churn Risk Analysis & Intervention Planning',
             color='#e6e8f4', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Fairness Analysis Across Segments

Verify that the model performs equitably across customer segments.
In production, this evaluation gate would block deployment if
AUC disparity exceeds the threshold.

In [ ]:
# Fairness check: AUC by segment
segments = {
    'Plan Type': 'plan_type',
    'Company Size': 'company_size'
}

print('FAIRNESS ANALYSIS')
print('=' * 60)

for segment_name, segment_col in segments.items():
    print(f'\n--- {segment_name} ---')
    group_aucs = []
    test_df = df.loc[X_test.index]

    for group_val in test_df[segment_col].unique():
        mask = test_df[segment_col] == group_val
        if mask.sum() >= 20 and y_test[mask].nunique() > 1:
            group_auc = roc_auc_score(y_test[mask], y_pred_proba[mask])
            group_aucs.append(group_auc)
            print(f'  {group_val:15s}: AUC = {group_auc:.4f} (n={mask.sum()})')

    if group_aucs:
        disparity = max(group_aucs) - min(group_aucs)
        status = 'PASS' if disparity <= 0.15 else 'FAIL'
        print(f'  Disparity: {disparity:.4f} [{status}]')

print(f'\n{"=" * 60}')
print('Fairness threshold: max AUC disparity <= 0.15')

## 11. GCP Vertex AI Pipeline Code (Reference)

The cells below show how this notebook's logic translates to
a production Vertex AI Pipeline with KFP v2 components.

In [ ]:
# ============================================================
# GCP VERTEX AI PIPELINE — REFERENCE CODE
# This code is shown for learning purposes. It requires
# a GCP project with Vertex AI and BigQuery configured.
# ============================================================

# --- Pipeline definition (KFP v2) ---
# from kfp.v2 import dsl, compiler
# from google.cloud import aiplatform
#
# @dsl.pipeline(
#     name="churn-prediction-pipeline",
#     description="End-to-end churn prediction with drift detection",
#     pipeline_root="gs://my-bucket/pipeline-root"
# )
# def churn_pipeline(
#     project_id: str = "my-project",
#     dataset_id: str = "churn_data",
#     prediction_date: str = "2026-03-06",
#     min_auc_threshold: float = 0.80
# ):
#     # Step 1: Validate source data
#     validation_task = validate_data(
#         project_id=project_id,
#         dataset_id=dataset_id
#     )
#
#     # Step 2: Feature engineering (conditional on validation)
#     with dsl.Condition(validation_task.output == True):
#         feature_task = engineer_features(
#             project_id=project_id,
#             dataset_id=dataset_id,
#             prediction_date=prediction_date
#         )
#
#         # Step 3: Train XGBoost
#         train_task = train_xgboost(
#             feature_dataset=feature_task.outputs["feature_dataset"],
#             target_column="churned"
#         )
#
#         # Step 4: Evaluate with fairness gate
#         eval_task = evaluate_model(
#             model_artifact=train_task.outputs["model_artifact"],
#             feature_dataset=feature_task.outputs["feature_dataset"],
#             min_auc_threshold=min_auc_threshold
#         )
#
#         # Step 5: Deploy only if evaluation passes
#         with dsl.Condition(eval_task.output == True):
#             deploy_task = deploy_model(
#                 model_artifact=train_task.outputs["model_artifact"],
#                 project_id=project_id
#             )
#
# # Compile pipeline to JSON
# compiler.Compiler().compile(
#     pipeline_func=churn_pipeline,
#     package_path="churn_pipeline.json"
# )
#
# # Submit to Vertex AI
# aiplatform.init(project="my-project", location="us-central1")
# job = aiplatform.PipelineJob(
#     display_name="churn-prediction-run",
#     template_path="churn_pipeline.json"
# )
# job.submit()

print('GCP Vertex AI Pipeline reference code (see comments above).')
print('Uncomment and configure with your GCP project to run in production.')

In [ ]:
# ============================================================
# VERTEX AI VIZIER — HYPERPARAMETER TUNING REFERENCE
# ============================================================

# from google.cloud import aiplatform
#
# # Define Vizier study for Bayesian optimization
# study = aiplatform.Study.create_or_load(
#     display_name="churn-xgboost-hpt",
#     problem_type="MAXIMIZE",
#     metric_id="auc_pr",
#     parameters=[
#         {"parameter_id": "max_depth",
#          "integer_value_spec": {"min_value": 3, "max_value": 10}},
#         {"parameter_id": "learning_rate",
#          "double_value_spec": {"min_value": 0.01, "max_value": 0.3},
#          "scale_type": "UNIT_LOG_SCALE"},
#         {"parameter_id": "n_estimators",
#          "integer_value_spec": {"min_value": 100, "max_value": 1000}},
#         {"parameter_id": "subsample",
#          "double_value_spec": {"min_value": 0.6, "max_value": 1.0}},
#         {"parameter_id": "colsample_bytree",
#          "double_value_spec": {"min_value": 0.6, "max_value": 1.0}},
#     ]
# )
#
# # Run 50 Bayesian optimization trials
# for trial_idx in range(50):
#     trial = study.suggest()
#     params = trial.parameters
#     # ... train model with suggested params ...
#     trial.complete(final_measurement={"auc_pr": auc_pr_score})

print('Vertex AI Vizier HPT reference code (see comments above).')

In [ ]:
# ============================================================
# MODEL DEPLOYMENT & BATCH PREDICTION REFERENCE
# ============================================================

# # Upload model to Vertex AI Model Registry
# model = aiplatform.Model.upload(
#     display_name="churn-predictor-v2",
#     artifact_uri="gs://my-bucket/models/churn/v2/",
#     serving_container_image_uri=(
#         "us-docker.pkg.dev/vertex-ai/prediction/"
#         "xgboost-cpu.1-7:latest"
#     ),
#     labels={"team": "retention", "env": "production"}
# )
#
# # Deploy to endpoint with canary traffic split
# endpoint = aiplatform.Endpoint.create(
#     display_name="churn-prediction-endpoint"
# )
# endpoint.deploy(
#     model=model,
#     machine_type="n1-standard-4",
#     min_replica_count=1,
#     max_replica_count=5,
#     traffic_percentage=10  # 10% canary
# )
#
# # Nightly batch prediction
# batch_job = model.batch_predict(
#     job_display_name="churn-batch-nightly",
#     instances_format="bigquery",
#     predictions_format="bigquery",
#     bigquery_source="bq://project.dataset.customer_features_latest",
#     bigquery_destination_prefix="bq://project.dataset.churn_predictions",
#     machine_type="n1-standard-8"
# )

print('Deployment & batch prediction reference code (see comments above).')

## 12. Summary

This notebook demonstrated an end-to-end customer churn prediction pipeline:

1. **Synthetic data generation** with realistic SaaS customer signals
2. **Feature engineering** with engagement scores, support health, billing risk
3. **XGBoost training** with hyperparameter tuning (GridSearchCV / Vizier)
4. **Comprehensive evaluation** including AUC-ROC, AUC-PR, calibration
5. **Feature importance** via built-in XGBoost and permutation methods
6. **Intervention scoring** with tiered priority ranking
7. **Fairness analysis** across customer segments
8. **GCP production code** for Vertex AI Pipelines, Vizier, and deployment

**Key GCP services:** Vertex AI Pipelines, Vizier, BigQuery, Model Registry, Model Monitoring

---
*CareerAlign GCP MLE Use Cases | [Back to Use Case 03 Page](../03-churn-prediction.html)*